In [1]:
import sys

In [2]:
%%capture
try:
    # Attempt to import a module that's only available in Colab
    from google.colab import drive

    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # Colab specific setup
    drive.mount("/content/drive")
    sys.path.append("/content/drive/MyDrive/structure-loss-classification/")
    my_local_data = "/content/drive/MyDrive/types/"
    %cd '/content/drive/MyDrive/structure-loss-classification/'
    %pip install scienceplots
    %pip install pytorch_lightning
else:
    # Local machine setup
    my_local_data = "/mnt/g/My Drive/types/"
    my_local_data_struct = "/mnt/g/My Drive/types-struct/"

In [3]:
import torch
import torch.nn as nn

import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision.models.feature_extraction import (
    get_graph_node_names,
    create_feature_extractor,
)

In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold

In [5]:
import pytorch_lightning as pl

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "notebook", "grid"])

In [7]:
import pickle

In [8]:
from models.models import LeNet5
from lightning_modules.lightning_modules import LitLeNet5
from visualization.filters import display_filters
from datasets.data_modules import CustomImageDataModule
from train.train import get_features, train_model

In [9]:
toTensorAndNormalize = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # mean  # std
    ]
)
aux_data = datasets.ImageFolder(root=my_local_data, transform=toTensorAndNormalize)

In [10]:
from datasets.datasets import CustomImageFolder

In [11]:
aux_data_struct = CustomImageFolder(
    root=my_local_data_struct,
    classification_mode="binary",
    transform=toTensorAndNormalize,
)

In [12]:
# Try to load cached targets first
try:
    with open(f"logdir/cached_targets_struct.pkl", "rb") as f:
        targets = pickle.load(f)
except FileNotFoundError:
    targets = [t for _, t in aux_data_struct]
    # Cache the targets for next time
    with open(f"logdir/cached_targets_struct.pkl", "wb") as f:
        pickle.dump(targets, f)

In [13]:
model2 = LitLeNet5(num_classes=2, size_layer_1=10, size_layer_2=5, learning_rate=0.001)

In [14]:
dataset = datasets.ImageFolder(my_local_data_struct)
print(f"Number of classes: {len(dataset.classes)}")
print(f"Class to index mapping: {dataset.class_to_idx}")

Number of classes: 2
Class to index mapping: {'badIngots': 0, 'goodIngots': 1}


In [15]:
kfold = StratifiedKFold(
    n_splits=4,
    shuffle=True,
)

In [16]:
validation_metrics = []

In [17]:
trainer_config = {
    "patience": 3,
    "accelerator": "gpu",
    "devices": -1,
    "max_epochs": 10,
    "precision": 32,
    "n_steps": 5,
}

In [18]:
# Assuming aux_data is a dataset object and targets are the labels
train_idx, val_idx, _, _ = train_test_split(
    range(len(aux_data_struct)), targets, test_size=0.2, random_state=42
)

train_data = torch.utils.data.Subset(aux_data_struct, train_idx)
val_data = torch.utils.data.Subset(aux_data_struct, val_idx)

In [19]:
data_module = CustomImageDataModule(
    train_dataset=train_data,
    val_dataset=val_data,
    batch_size=16,
    num_workers=4,
)

In [20]:
# Assuming model2 is initialized, and trainer_config is defined
# val_metrics = train_model(
#     model=model2,
#     trainer_config=trainer_config,
#     save_dir="logdir-struct/",
#     data_module=data_module,
# )

In [21]:
default_config = {
    "layer_1_size": 128,
    "layer_2_size": 256,
    "lr": 1e-3,
    "batch_size": 16
}

In [22]:
model_ray = LitLeNet5(num_classes=2,
                      size_layer_1=default_config['layer_1_size'],
                      size_layer_2=default_config['layer_2_size'],
                      learning_rate=default_config['lr'])

In [25]:
from ray.train.lightning import (
    RayDDPStrategy,
    RayLightningEnvironment,
    RayTrainReportCallback,
    prepare_trainer,
)

In [27]:
def train_func(config):
    dm = CustomImageDataModule(train_dataset= train_data,
                               val_dataset=val_data,
                               batch_size=config["batch_size"],
                               num_workers = 12)
    
    model = LitLeNet5(num_classes=2,
                      size_layer_1=default_config['layer_1_size'],
                      size_layer_2=default_config['layer_2_size'],
                      learning_rate=default_config['lr'])

    num_steps_per_epoch = max(1, (len(train_data) + len(val_data)) // config["batch_size"])

    trainer = pl.Trainer(
        devices="auto",
        accelerator="auto",
        strategy=RayDDPStrategy(),
        callbacks=[RayTrainReportCallback()],
        plugins=[RayLightningEnvironment()],
        enable_progress_bar=False,
        max_epochs=2,
        log_every_n_steps=min(50, num_steps_per_epoch)
    )
    trainer = prepare_trainer(trainer)
    trainer.fit(model, datamodule=dm)

In [28]:
import ray

In [29]:
def auto_init_ray():
    num_gpus = torch.cuda.device_count()
    resources = {'num_cpus': 12, 'num_gpus': None}

    if num_gpus > 0:
        resources['num_gpus'] = num_gpus

    ray.init(**resources)

auto_init_ray()

2024-02-16 16:32:48,956	INFO worker.py:1673 -- Started a local Ray instance.


In [30]:
from ray import tune
from ray.tune.schedulers import ASHAScheduler

In [31]:
resources = ray.cluster_resources()

In [32]:
search_space = {
    "layer_1_size": tune.choice([32, 64, 128]),
    "layer_2_size": tune.choice([64, 128, 256]),
    "lr": tune.loguniform(1e-5, 1e-2),
    "batch_size": tune.choice([32, 64]),
}

In [33]:
# The maximum training epochs
num_epochs = 2

# Number of sampls from parameter space
num_samples = 3

In [34]:
scheduler = ASHAScheduler(max_t=num_epochs, grace_period=1, reduction_factor=2)

In [35]:
resources['CPU']

12.0

In [36]:
from ray.train import RunConfig, ScalingConfig, CheckpointConfig

scaling_config = ScalingConfig(
    num_workers=1,
    use_gpu=True,
    trainer_resources={"CPU":resources['CPU']}
)

run_config = RunConfig(
    checkpoint_config=CheckpointConfig(
        num_to_keep=2,
        checkpoint_score_attribute="val_accuracy",
        checkpoint_score_order="max",
    ),
)

In [37]:
from ray.train.torch import TorchTrainer

# Define a TorchTrainer without hyper-parameters for Tuner
ray_trainer = TorchTrainer(
    train_func,
    scaling_config=scaling_config,
    run_config=run_config,
)

In [38]:
def tune_mnist_asha(num_samples=10):
    scheduler = ASHAScheduler(max_t=num_epochs, grace_period=1, reduction_factor=2)

    tuner = tune.Tuner(
        ray_trainer,
        param_space={"train_loop_config": search_space},
        tune_config=tune.TuneConfig(
            metric="val_accuracy",
            mode="max",
            num_samples=num_samples,
            scheduler=scheduler,
        ),
    )
    return tuner.fit()

In [39]:
results = tune_mnist_asha(num_samples=num_samples)

(TorchTrainer pid=1069) Starting distributed worker processes: ['1210 (172.25.160.153)']
(RayTrainWorker pid=1210) Setting up process group for: env:// [rank=0, world_size=1]
(RayTrainWorker pid=1210) GPU available: True (cuda), used: True
(RayTrainWorker pid=1210) TPU available: False, using: 0 TPU cores
(RayTrainWorker pid=1210) IPU available: False, using: 0 IPUs
(RayTrainWorker pid=1210) HPU available: False, using: 0 HPUs
(RayTrainWorker pid=1210) Missing logger folder: /home/alfredosg/ray_results/TorchTrainer_2024-02-16_16-32-50/TorchTrainer_a4c46_00000_0_batch_size=32,layer_1_size=128,layer_2_size=64,lr=0.0009_2024-02-16_16-32-50/lightning_logs
(RayTrainWorker pid=1210) LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
(RayTrainWorker pid=1210) 
(RayTrainWorker pid=1210)   | Name     | Type               | Params
(RayTrainWorker pid=1210) ------------------------------------------------
(RayTrainWorker pid=1210) 0 | accuracy | MulticlassAccuracy | 0     
(RayTrainWorker pid=1210) 1 | f1

2024-02-16 16:41:14,836	INFO tune.py:1047 -- Total run time: 504.43 seconds (504.40 seconds for the tuning loop).


In [ ]:
results.get_best_result(metric='val_accuracy',
                        mode='max')

In [ ]:
results.get_best_result(metric='val_accuracy',
                        mode='max').config